In [6]:
# ============================================================
# 0. Imports and frozen experimental configuration
# ============================================================
!apt-get -qq update
!apt-get -qq install -y libreoffice-writer antiword > /dev/null
!pip -q install pymupdf pymupdf4llm

import json
import hashlib
import platform
import re
import shutil
import subprocess
import sys
import unicodedata

from collections import Counter
from datetime import datetime
from pathlib import Path

import fitz
import pymupdf4llm

from google.colab import files

DOCUMENT_ID = "D8"
DOCUMENT_NAME = (
    "World Bank — Bhutan - Land Management Project — "
    "Project Information Document (PID), Concept Stage"
)

BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"
PARENT_BRANCH = "B"

SOURCE_FORMAT = ".doc"
EXPECTED_SOURCE_SHA256 = (
    "61aacfd3138ecfba59fac51d29a970de45d8a909c74b744e756e8a283666c7b5"
)
EXPECTED_PHYSICAL_PAGE_COUNT = 4

EXPECTED_BRANCH_B_REPRESENTATION_SHA256 = (
    "60092e846b5ead54cf7aa11463b8d2753045d03133238d002f4c871133d33a74"
)

# ------------------------------------------------------------
# Frozen Stage 1 expectations.
# Used only AFTER extraction for diagnostics / Stage 4 validation.
# They are NOT disclosed to the model.
# ------------------------------------------------------------

EXPECTED_RECORD_COUNT = 49

EXPECTED_CATEGORY_COUNTS = {
    "Project metadata": 13,
    "Development issue": 10,
    "Bank rationale": 2,
    "Project objective": 3,
    "Project component": 3,
    "Safeguard policy": 6,
    "Financing": 7,
    "Contact information": 5
}

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location"
]

STRING_OR_NULL_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location"
]

MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Source Location"
]

ALLOWED_CATEGORIES = set(EXPECTED_CATEGORY_COUNTS)

EXPECTED_QUALIFIER_VALUES = {
    "at least",
    "another",
    "less than",
    "some",
    "up to"
}

REQUIRED_SECTION_MARKERS = {
    "header":
        "PROJECT INFORMATION DOCUMENT (PID)",
    "concept_stage":
        "CONCEPT STAGE",
    "development_issues":
        "1. Key development issues and rationale for Bank involvement",
    "objectives":
        "2. Proposed objective(s)",
    "description":
        "3. Preliminary description",
    "safeguards":
        "4. Safeguard Policies that Might Apply",
    "financing":
        "5. Tentative financing",
    "contact":
        "6. Contact point"
}

REQUIRED_SOURCE_MARKERS = [
    "AB526",
    "P087039",
    "Indigenous Pelple",
    "BORROWER/RECEPIENT",
    "F1",
    "one-quarter",
    "one-third",
    "16.5"
]

OUTPUT_DIR = Path("outputs_D8_branch_C")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONVERSION_DIR = OUTPUT_DIR / "conversion"
CONVERSION_DIR.mkdir(parents=True, exist_ok=True)

INTERMEDIATE_PDF_PATH = (
    CONVERSION_DIR / "D8_branch_C_parent_B_intermediate.pdf"
)

PARENT_CHECK_PATH = (
    OUTPUT_DIR / "D8_branch_C_parent_B_equivalence_check.json"
)

NORMALISATION_CHECK_PATH = (
    OUTPUT_DIR / "D8_branch_C_normalisation_check.json"
)

REPRESENTATION_PATH = (
    OUTPUT_DIR / "D8_branch_C_normalised_markdown.md"
)

REPRESENTATION_METADATA_PATH = (
    OUTPUT_DIR / "D8_branch_C_representation_metadata.json"
)

PROMPT_PATH = (
    OUTPUT_DIR / "D8_branch_C_prompt.txt"
)

EXPERIMENT_METADATA_PRE_PATH = (
    OUTPUT_DIR / "D8_branch_C_experiment_metadata_pre.json"
)

PRECHECK_PATH = (
    OUTPUT_DIR / "D8_branch_C_pre_extraction_check.json"
)

RAW_RESPONSE_PATH = (
    OUTPUT_DIR / "D8_branch_C_raw_response.txt"
)

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR / "D8_branch_C_parsed_extraction.json"
)

STRUCTURE_CHECK_PATH = (
    OUTPUT_DIR / "D8_branch_C_structure_check.json"
)

EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR / "D8_branch_C_experiment_metadata.json"
)

EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR / "D8_branch_C_experiment_summary.json"
)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Parent branch:", PARENT_BRANCH)
print("Expected physical pages:", EXPECTED_PHYSICAL_PAGE_COUNT)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Document: D8
Branch: C
Parent branch: B
Expected physical pages: 4


In [7]:
# ============================================================
# 1. Upload original D8 DOC and required Branch B artefacts
# ============================================================
# Upload exactly:
#   1) original D8 legacy DOC
#   2) D8_branch_B_structural_markdown.md
#   3) D8_branch_B_conversion_integrity.json

uploaded = files.upload()
names = list(uploaded.keys())

doc_files = [
    Path(name)
    for name in names
    if name.lower().endswith(".doc")
]

md_files = [
    Path(name)
    for name in names
    if name.lower().endswith(".md")
]

json_files = [
    Path(name)
    for name in names
    if name.lower().endswith(".json")
]

if (
    len(doc_files) != 1
    or len(md_files) != 1
    or len(json_files) != 1
):
    raise ValueError(
        "Upload exactly one DOC, one Branch B structural Markdown file, "
        "and one Branch B conversion-integrity JSON file."
    )

SOURCE_PATH = doc_files[0]
BRANCH_B_REPRESENTATION_PATH = md_files[0]
BRANCH_B_CHECK_PATH = json_files[0]

print("Source:", SOURCE_PATH.name)
print("Branch B representation:", BRANCH_B_REPRESENTATION_PATH.name)
print("Branch B integrity:", BRANCH_B_CHECK_PATH.name)


Saving D8_branch_B_conversion_integrity.json to D8_branch_B_conversion_integrity.json
Saving D8_branch_B_structural_markdown.md to D8_branch_B_structural_markdown.md
Saving D8 - Project0Inform1ment010Concept0Stage.doc to D8 - Project0Inform1ment010Concept0Stage.doc
Source: D8 - Project0Inform1ment010Concept0Stage.doc
Branch B representation: D8_branch_B_structural_markdown.md
Branch B integrity: D8_branch_B_conversion_integrity.json


In [8]:
# ============================================================
# 2. Verify frozen source identity and Branch B parent integrity
# ============================================================
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(chunk_size),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def sha256_text(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


if SOURCE_PATH.suffix.lower() != SOURCE_FORMAT:
    raise ValueError("Unexpected D8 source format.")

SOURCE_SHA256 = sha256_file(SOURCE_PATH)

SOURCE_HASH_MATCH = (
    SOURCE_SHA256
    == EXPECTED_SOURCE_SHA256
)

if not SOURCE_HASH_MATCH:
    raise ValueError(
        "Uploaded D8 DOC does not match the frozen Stage 1 source identity."
    )

source_bytes = SOURCE_PATH.read_bytes()

OLE_SIGNATURE = bytes.fromhex(
    "D0CF11E0A1B11AE1"
)

LEGACY_BINARY_DOC = source_bytes.startswith(
    OLE_SIGNATURE
)

if not LEGACY_BINARY_DOC:
    raise ValueError(
        "Uploaded D8 source does not appear to be the expected legacy binary DOC."
    )


with open(
    BRANCH_B_CHECK_PATH,
    "r",
    encoding="utf-8"
) as f:
    branch_b_check = json.load(f)


if branch_b_check.get("document_id") != DOCUMENT_ID:
    raise ValueError(
        "Branch B integrity artefact belongs to another document."
    )

if branch_b_check.get("branch") != "B":
    raise ValueError(
        "Uploaded integrity artefact is not from Branch B."
    )

if branch_b_check.get("source_sha256") != SOURCE_SHA256:
    raise ValueError(
        "Branch B parent representation was generated from another D8 source identity."
    )

if not branch_b_check.get(
    "conversion_integrity_passed",
    False
):
    raise ValueError(
        "Branch B parent representation did not pass conversion integrity."
    )


SOURCE_B_MARKDOWN = (
    BRANCH_B_REPRESENTATION_PATH.read_text(
        encoding="utf-8"
    )
)

if not SOURCE_B_MARKDOWN.strip():
    raise ValueError(
        "Uploaded Branch B structural Markdown is empty."
    )

SOURCE_B_SHA256 = sha256_text(
    SOURCE_B_MARKDOWN
)

print("Frozen source identity verified.")
print("Legacy DOC signature verified.")
print("Branch B conversion integrity verified.")
print("Branch B SHA-256:", SOURCE_B_SHA256)


Frozen source identity verified.
Legacy DOC signature verified.
Branch B conversion integrity verified.
Branch B SHA-256: 60092e846b5ead54cf7aa11463b8d2753045d03133238d002f4c871133d33a74


In [9]:
# ============================================================
# 3. Verify the frozen Branch B representation artefact directly
# ============================================================
#
# D8 is a legacy binary DOC. Re-rendering the same DOC through
# LibreOffice can produce a different intermediate PDF across runs
# or environments (for example because of renderer/version/font
# differences or PDF metadata), even when the visible/source content
# is unchanged.
#
# Therefore Branch C must NOT regenerate Branch B and compare the
# regenerated Markdown byte-for-byte.
#
# Instead, Branch C starts from the already frozen Branch B
# representation artefact and verifies:
#
#   1. the original D8 source identity is unchanged;
#   2. the uploaded Branch B integrity artefact belongs to D8 and passed;
#   3. the uploaded Branch B Markdown has the exact SHA-256 recorded
#      for the final frozen Branch B representation;
#   4. the four page boundaries and Branch B structural financing
#      reconstruction are still present.
#
# This provides the correct B -> C experimental lineage:
#
#   frozen Branch B representation
#       -> deterministic Branch C normalisation
#
# No second DOC -> PDF -> Markdown conversion is introduced here.
# ============================================================

UPLOADED_BRANCH_B_SHA256 = sha256_text(
    SOURCE_B_MARKDOWN
)

BRANCH_B_HASH_MATCH = (
    UPLOADED_BRANCH_B_SHA256
    == EXPECTED_BRANCH_B_REPRESENTATION_SHA256
)

if not BRANCH_B_HASH_MATCH:
    raise ValueError(
        "Uploaded Branch B Markdown does not match the frozen final "
        "D8 Branch B representation SHA-256."
    )


# ------------------------------------------------------------
# Verify page boundaries in the frozen Branch B representation
# ------------------------------------------------------------

BRANCH_B_PAGE_PATTERN = re.compile(
    r"^## Source Page (\d+)$",
    flags=re.MULTILINE
)

branch_b_page_markers = BRANCH_B_PAGE_PATTERN.findall(
    SOURCE_B_MARKDOWN
)

expected_branch_b_pages = [
    str(page_number)
    for page_number in range(
        1,
        EXPECTED_PHYSICAL_PAGE_COUNT + 1
    )
]

BRANCH_B_PAGE_SEQUENCE_VALID = (
    branch_b_page_markers
    == expected_branch_b_pages
)


# ------------------------------------------------------------
# Verify inherited Branch B financing reconstruction
# ------------------------------------------------------------

EXPECTED_RECONSTRUCTED_FINANCING_LABELS = [
    "LOCAL GOVTS. (PROV., DISTRICT, CITY) OF BORROWING COUNTRY",
    "NON-GOVERNMENT ORGANIZATION (NGO) OF BORROWING COUNTRY"
]

branch_b_financing_label_checks = {
    label: (
        label
        in SOURCE_B_MARKDOWN
    )
    for label
    in EXPECTED_RECONSTRUCTED_FINANCING_LABELS
}

BRANCH_B_FINANCING_STRUCTURE_VALID = all(
    branch_b_financing_label_checks.values()
)

FINANCING_LABEL_RECONSTRUCTION_COUNT = int(
    branch_b_check.get(
        "financing_label_reconstruction_count",
        0
    )
)

if FINANCING_LABEL_RECONSTRUCTION_COUNT != 2:
    raise ValueError(
        "The frozen Branch B integrity artefact does not report the "
        "expected two financing-label reconstruction events."
    )

if not BRANCH_B_PAGE_SEQUENCE_VALID:
    raise ValueError(
        "Frozen Branch B representation does not preserve the expected "
        "four source-page boundaries."
    )

if not BRANCH_B_FINANCING_STRUCTURE_VALID:
    raise ValueError(
        "Frozen Branch B representation does not contain both expected "
        "reconstructed financing labels."
    )

print(
    "Frozen Branch B representation SHA-256:",
    UPLOADED_BRANCH_B_SHA256
)

print(
    "Frozen Branch B hash verified:",
    BRANCH_B_HASH_MATCH
)

print(
    "Branch B page sequence verified:",
    BRANCH_B_PAGE_SEQUENCE_VALID
)

print(
    "Inherited financing reconstruction count:",
    FINANCING_LABEL_RECONSTRUCTION_COUNT
)

Frozen Branch B representation SHA-256: 60092e846b5ead54cf7aa11463b8d2753045d03133238d002f4c871133d33a74
Frozen Branch B hash verified: True
Branch B page sequence verified: True
Inherited financing reconstruction count: 2


In [10]:
# ============================================================
# 4. Verify frozen Branch B parent equivalence
# ============================================================
#
# For D8, parent equivalence means that Branch C receives the exact
# frozen Branch B representation artefact used in the completed
# Branch B experiment.
#
# We intentionally do NOT require a newly rendered LibreOffice PDF
# to reproduce the same hash because that conversion layer is not
# byte-deterministic across runs/environments.
# ============================================================

PARENT_EQUIVALENCE_PASSED = bool(
    SOURCE_HASH_MATCH
    and branch_b_check.get(
        "conversion_integrity_passed",
        False
    )
    and BRANCH_B_HASH_MATCH
    and BRANCH_B_PAGE_SEQUENCE_VALID
    and BRANCH_B_FINANCING_STRUCTURE_VALID
    and FINANCING_LABEL_RECONSTRUCTION_COUNT == 2
)

parent_check = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "source_sha256":
        SOURCE_SHA256,

    "source_hash_matches_frozen_identity":
        SOURCE_HASH_MATCH,

    "branch_B_conversion_integrity_passed":
        bool(
            branch_b_check.get(
                "conversion_integrity_passed",
                False
            )
        ),

    "expected_frozen_branch_B_sha256":
        EXPECTED_BRANCH_B_REPRESENTATION_SHA256,

    "uploaded_branch_B_sha256":
        UPLOADED_BRANCH_B_SHA256,

    "uploaded_branch_B_hash_matches_frozen_parent":
        BRANCH_B_HASH_MATCH,

    "branch_B_page_markers":
        branch_b_page_markers,

    "branch_B_page_sequence_valid":
        BRANCH_B_PAGE_SEQUENCE_VALID,

    "branch_B_financing_label_checks":
        branch_b_financing_label_checks,

    "branch_B_financing_structure_valid":
        BRANCH_B_FINANCING_STRUCTURE_VALID,

    "financing_label_reconstruction_count":
        FINANCING_LABEL_RECONSTRUCTION_COUNT,

    "parent_equivalence_method":
        (
            "Frozen Branch B artefact SHA-256 + Branch B integrity "
            "provenance; no DOC re-rendering in Branch C"
        ),

    "branch_B_regeneration_attempted":
        False,

    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED
}

PARENT_CHECK_PATH.write_text(
    json.dumps(
        parent_check,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        parent_check,
        ensure_ascii=False,
        indent=2
    )
)

if not PARENT_EQUIVALENCE_PASSED:
    raise ValueError(
        "D8 Branch C parent-equivalence verification failed. "
        "The uploaded Branch B representation must be the exact frozen "
        "Branch B artefact used in the completed Branch B experiment."
    )

{
  "document_id": "D8",
  "branch": "C",
  "parent_branch": "B",
  "source_sha256": "61aacfd3138ecfba59fac51d29a970de45d8a909c74b744e756e8a283666c7b5",
  "source_hash_matches_frozen_identity": true,
  "branch_B_conversion_integrity_passed": true,
  "expected_frozen_branch_B_sha256": "60092e846b5ead54cf7aa11463b8d2753045d03133238d002f4c871133d33a74",
  "uploaded_branch_B_sha256": "60092e846b5ead54cf7aa11463b8d2753045d03133238d002f4c871133d33a74",
  "uploaded_branch_B_hash_matches_frozen_parent": true,
  "branch_B_page_markers": [
    "1",
    "2",
    "3",
    "4"
  ],
  "branch_B_page_sequence_valid": true,
  "branch_B_financing_label_checks": {
    "LOCAL GOVTS. (PROV., DISTRICT, CITY) OF BORROWING COUNTRY": true,
    "NON-GOVERNMENT ORGANIZATION (NGO) OF BORROWING COUNTRY": true
  },
  "branch_B_financing_structure_valid": true,
  "financing_label_reconstruction_count": 2,
  "parent_equivalence_method": "Frozen Branch B artefact SHA-256 + Branch B integrity provenance; no DOC re-ren

In [11]:
# ============================================================
# 5. Define conservative deterministic Branch C normalisation
# ============================================================
#
# Following D1–D7, Branch C changes representation only.
#
# Allowed:
# - Unicode NFKC
# - Unicode-space standardisation
# - typographic apostrophe standardisation
# - dash/minus-glyph standardisation
# - soft-hyphen removal
# - line-ending standardisation
# - horizontal whitespace normalisation
# - excessive blank-line standardisation
#
# Not applied:
# - page filtering/removal
# - semantic rewriting/harmonisation
# - source-code/spelling repair
# - unit conversion
# - numerical calculation
# - manual correction
# - reference-guided reconstruction
# ============================================================

UNICODE_SPACE_CHARACTERS = [
    "\u00a0", "\u1680", "\u2000", "\u2001", "\u2002",
    "\u2003", "\u2004", "\u2005", "\u2006", "\u2007",
    "\u2008", "\u2009", "\u200a", "\u202f", "\u205f",
    "\u3000"
]

APOSTROPHE_REPLACEMENTS = {
    "’": "'",
    "‘": "'",
    "‛": "'",
    "´": "'",
    "`": "'"
}

DASH_REPLACEMENTS = {
    "‐": "-",
    "‑": "-",
    "‒": "-",
    "–": "-",
    "—": "-",
    "−": "-"
}


def normalise_text_representation(text):
    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    for character in UNICODE_SPACE_CHARACTERS:
        text = text.replace(
            character,
            " "
        )

    for source, target in APOSTROPHE_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    for source, target in DASH_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    text = text.replace(
        "\u00ad",
        ""
    )

    text = (
        text
        .replace("\r\n", "\n")
        .replace("\r", "\n")
    )

    lines = []

    for line in text.splitlines():
        line = re.sub(
            r"[ \t\f\v]+",
            " ",
            line
        ).rstrip()

        lines.append(
            line
        )

    text = "\n".join(
        lines
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return (
        text.strip()
        + "\n"
    )


In [12]:
# ============================================================
# 6. Apply Branch C normalisation to the COMPLETE Branch B representation
# ============================================================
NORMALISED_MARKDOWN = (
    normalise_text_representation(
        SOURCE_B_MARKDOWN
    )
)

if not NORMALISED_MARKDOWN.strip():
    raise ValueError(
        "D8 Branch C normalisation produced an empty representation."
    )

print(
    "Branch B characters:",
    len(SOURCE_B_MARKDOWN)
)

print(
    "Branch C characters:",
    len(NORMALISED_MARKDOWN)
)


Branch B characters: 11442
Branch C characters: 11391


In [13]:
# ============================================================
# 7. Verify Branch C normalisation integrity
# ============================================================
#
# Transformation-aware principle:
# raw-string equality is not required because Unicode/spacing
# cleanup is the intended Branch C intervention.
#
# Integrity is established by proving that:
# 1. the exact Branch B parent is known;
# 2. all four physical-page boundaries remain;
# 3. the declared normalisation function exactly reproduces C;
# 4. required sections/source spellings/codes remain;
# 5. quantitative source forms remain after canonicalising only
#    transformations that Branch C is explicitly allowed to make.
# ============================================================


# ------------------------------------------------------------
# A. Physical page sequence
# ------------------------------------------------------------

PAGE_PATTERN = re.compile(
    r"^## Source Page (\d+)$",
    flags=re.MULTILINE
)

parent_pages = PAGE_PATTERN.findall(
    SOURCE_B_MARKDOWN
)

branch_c_pages = PAGE_PATTERN.findall(
    NORMALISED_MARKDOWN
)

expected_pages = [
    str(page_number)
    for page_number
    in range(
        1,
        EXPECTED_PHYSICAL_PAGE_COUNT + 1
    )
]

page_sequence_preserved = (
    parent_pages
    == branch_c_pages
    == expected_pages
)


# ------------------------------------------------------------
# B. Exact deterministic transformation reproducibility
# ------------------------------------------------------------

EXPECTED_NORMALISED_MARKDOWN = (
    normalise_text_representation(
        SOURCE_B_MARKDOWN
    )
)

deterministic_representation_verified = (
    NORMALISED_MARKDOWN
    == EXPECTED_NORMALISED_MARKDOWN
)


# ------------------------------------------------------------
# C. Required section markers
# ------------------------------------------------------------

canonical_representation = (
    NORMALISED_MARKDOWN
    .casefold()
)

section_marker_checks = {}

for key, marker in REQUIRED_SECTION_MARKERS.items():
    canonical_marker = (
        normalise_text_representation(
            marker
        )
        .strip()
        .casefold()
    )

    section_marker_checks[key] = (
        canonical_marker
        in canonical_representation
    )

all_section_markers_preserved = all(
    section_marker_checks.values()
)


# ------------------------------------------------------------
# D. Source spellings / codes / textual fractions
# ------------------------------------------------------------

source_marker_checks = {}

for marker in REQUIRED_SOURCE_MARKERS:
    canonical_marker = (
        normalise_text_representation(
            marker
        )
        .strip()
        .casefold()
    )

    source_marker_checks[marker] = (
        canonical_marker
        in canonical_representation
    )

all_source_markers_preserved = all(
    source_marker_checks.values()
)


# ------------------------------------------------------------
# E. Explicitly verify the two reconstructed financing labels
# ------------------------------------------------------------

financing_label_checks = {
    "LOCAL GOVTS. (PROV., DISTRICT, CITY) OF BORROWING COUNTRY":
        (
            "local govts. (prov., district, city) of borrowing country"
            in canonical_representation
        ),

    "NON-GOVERNMENT ORGANIZATION (NGO) OF BORROWING COUNTRY":
        (
            "non-government organization (ngo) of borrowing country"
            in canonical_representation
        )
}

all_financing_labels_preserved = all(
    financing_label_checks.values()
)


# ------------------------------------------------------------
# F. Transformation-aware quantitative-token preservation
# ------------------------------------------------------------

VALUE_PATTERNS = {
    "percentages":
        r"(?<![\w])\d+(?:\.\d+)?\s*%",

    "currency_million_expressions":
        r"\$\s*\d+(?:\.\d+)?\s*million\b",

    "plain_decimal_or_integer_values":
        r"(?<![\w.])\d+(?:\.\d+)?(?![\w.])",

    "calendar_dates":
        (
            r"\b(?:January|February|March|April|May|June|July|"
            r"August|September|October|November|December)\s+"
            r"\d{1,2},\s+\d{4}\b"
        )
}


def canonicalise_value_token(token):
    token = (
        normalise_text_representation(
            token
        )
        .strip()
    )

    token = re.sub(
        r"[ \t]+",
        " ",
        token
    )

    token = re.sub(
        r"\$\s+",
        "$",
        token
    )

    return token.casefold()


numeric_preservation = {}

for label, pattern in VALUE_PATTERNS.items():

    before = [
        canonicalise_value_token(
            token
        )
        for token
        in re.findall(
            pattern,
            SOURCE_B_MARKDOWN,
            flags=re.IGNORECASE
        )
    ]

    after = [
        canonicalise_value_token(
            token
        )
        for token
        in re.findall(
            pattern,
            NORMALISED_MARKDOWN,
            flags=re.IGNORECASE
        )
    ]

    before_counter = Counter(
        before
    )

    after_counter = Counter(
        after
    )

    missing = list(
        (
            before_counter
            - after_counter
        ).elements()
    )

    added = list(
        (
            after_counter
            - before_counter
        ).elements()
    )

    numeric_preservation[label] = {
        "count_before":
            len(before),

        "count_after":
            len(after),

        "missing_token_count":
            len(missing),

        "added_token_count":
            len(added),

        "passed":
            (
                len(missing) == 0
                and len(added) == 0
            )
    }


numeric_values_preserved = all(
    result["passed"]
    for result
    in numeric_preservation.values()
)


# ------------------------------------------------------------
# G. Complete-source-retention decision
# ------------------------------------------------------------

normalisation_integrity_passed = bool(
    PARENT_EQUIVALENCE_PASSED
    and page_sequence_preserved
    and deterministic_representation_verified
    and all_section_markers_preserved
    and all_source_markers_preserved
    and all_financing_labels_preserved
    and numeric_values_preserved
)


normalisation_check = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "parent_page_markers":
        parent_pages,

    "branch_C_page_markers":
        branch_c_pages,

    "page_sequence_preserved":
        page_sequence_preserved,

    "deterministic_representation_verified":
        deterministic_representation_verified,

    "section_marker_checks":
        section_marker_checks,

    "all_section_markers_preserved":
        all_section_markers_preserved,

    "source_marker_checks":
        source_marker_checks,

    "all_source_markers_preserved":
        all_source_markers_preserved,

    "financing_label_checks":
        financing_label_checks,

    "all_financing_labels_preserved":
        all_financing_labels_preserved,

    "numeric_token_preservation":
        numeric_preservation,

    "numeric_values_preserved":
        numeric_values_preserved,

    "complete_4_page_representation_retained":
        True,

    "source_scope_filtering_applied":
        False,

    "page_removal_applied":
        False,

    "page_cropping_applied":
        False,

    "ocr_applied":
        False,

    "structural_financing_label_reconstruction_inherited_from_branch_B":
        True,

    "financing_label_reconstruction_count":
        FINANCING_LABEL_RECONSTRUCTION_COUNT,

    "unicode_nfkc_normalisation_applied":
        True,

    "unicode_space_standardisation_applied":
        True,

    "apostrophe_standardisation_applied":
        True,

    "dash_and_minus_standardisation_applied":
        True,

    "soft_hyphen_removal_applied":
        True,

    "line_endings_standardised":
        True,

    "horizontal_whitespace_normalisation_applied":
        True,

    "paragraph_line_merging_applied":
        False,

    "line_break_hyphenation_repair_applied":
        False,

    "semantic_harmonisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "source_spelling_repair_applied":
        False,

    "unit_conversion_applied":
        False,

    "numeric_calculation_applied":
        False,

    "manual_correction_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "normalisation_integrity_passed":
        normalisation_integrity_passed
}


NORMALISATION_CHECK_PATH.write_text(
    json.dumps(
        normalisation_check,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        normalisation_check,
        ensure_ascii=False,
        indent=2
    )
)

if not normalisation_integrity_passed:
    raise ValueError(
        "D8 Branch C normalisation-integrity checks failed. "
        "Inspect parent equivalence, page/section preservation, "
        "source spellings/codes, financing labels and quantitative tokens."
    )


{
  "document_id": "D8",
  "branch": "C",
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "parent_page_markers": [
    "1",
    "2",
    "3",
    "4"
  ],
  "branch_C_page_markers": [
    "1",
    "2",
    "3",
    "4"
  ],
  "page_sequence_preserved": true,
  "deterministic_representation_verified": true,
  "section_marker_checks": {
    "header": true,
    "concept_stage": true,
    "development_issues": true,
    "objectives": true,
    "description": true,
    "safeguards": true,
    "financing": true,
    "contact": true
  },
  "all_section_markers_preserved": true,
  "source_marker_checks": {
    "AB526": true,
    "P087039": true,
    "Indigenous Pelple": true,
    "BORROWER/RECEPIENT": true,
    "F1": true,
    "one-quarter": true,
    "one-third": true,
    "16.5": true
  },
  "all_source_markers_preserved": true,
  "financing_label_checks": {
    "LOCAL GOVTS. (PROV., DISTRICT, CITY) OF BORROWING COUNTRY": true,
    "NON-GOVERNMENT ORGANIZATION (NGO) OF BORROWI

In [14]:
# ============================================================
# 8. Save Branch C representation
# ============================================================
REPRESENTATION_PATH.write_text(
    NORMALISED_MARKDOWN,
    encoding="utf-8"
)

REPRESENTATION_SHA256 = sha256_file(
    REPRESENTATION_PATH
)

print(
    "Saved:",
    REPRESENTATION_PATH.name
)

print(
    "Representation SHA-256:",
    REPRESENTATION_SHA256
)


Saved: D8_branch_C_normalised_markdown.md
Representation SHA-256: 9c0d72c972748af919626e4b4d0c47893c2531ecef6e90ac8572efe102f09193


In [15]:
# ============================================================
# 9. Create controlled Branch C extraction prompt
# ============================================================
#
# Same substantive scope/schema as final Branch B.
# The expected 49-record/category counts are NOT disclosed.
# ============================================================

BRANCH_C_PROMPT = """You are an information extraction assistant.

Extract the project-information records represented within the defined
scope of the attached deterministically normalised structural Markdown
representation of the original legacy Word document:

“Bhutan - Land Management Project”
Project Information Document (PID), Concept Stage.

Treat the attached deterministically normalised structural Markdown
document as the only source of information.

Include records from the following defined source regions.

1. Project metadata

Extract one record for each of these labelled header fields:

- Report No.
- Project Name
- Region
- Sector
- Project ID
- GEF Focal Area
- Borrower(s)
- Implementing Agency
- Environment Category
- Safeguard Classification
- Date PID Prepared
- Estimated Date of Appraisal Authorization
- Estimated Date of Board Approval

For the Sector field, preserve the complete represented sector text as
a single Value. Do not split its embedded percentages into additional
records.

For checkbox fields, extract the selected category represented by the
document.

Do not create separate records from implementing-agency telephone
numbers embedded within the Implementing Agency field.

2. Development issues

Within Section 1, “Key development issues and rationale for Bank
involvement”, extract the predefined quantitative development
observations concerning:

- the long-term forest-cover policy requirement;
- the share of country area set aside for protected areas;
- the additional area offered for wildlife corridors;
- population density per square kilometre of arable land;
- the urban growth rate;
- arable land as a share of land area;
- agricultural land affected by water erosion;
- the contribution of hydropower revenue to the development budget;
- villages not connected to feeder roads;
- villages facing food insecurity.

Do not extract comparison values concerning other world regions as
separate observations.

3. Bank rationale

Within the “Rationale for Bank Involvement” subsection, extract the
principal qualitative records concerning:

- limitations of the existing sector-oriented institutional framework
  in providing cross-sectoral accountability and incentive mechanisms;
- the governance, political-will and environmental-stewardship factors
  supporting Bhutan's suitability for GEF support.

Do not create additional records from illustrative examples or
supporting narrative details.

4. Project objectives

Within Section 2, “Proposed objective(s)”, extract each principal
project-objective statement represented by the source.

The scope consists of the objective to promote sustainable-land-
management mechanisms, the objective concerning technical innovations,
ecosystem functions and cross-sectoral mechanisms, and the objective
concerning multi-sectoral land and watershed planning with local
participation.

5. Project components

Within Section 3, “Preliminary description”, extract one record for
each explicitly labelled project component.

For each component:

- preserve the component number and component name;
- preserve a concise source-grounded description of the component;
- extract the explicitly represented estimated cost as Value;
- preserve the represented monetary scale in Unit;
- preserve any explicit approximation, ceiling or threshold wording
  associated with the cost in Qualifier.

Do not calculate a total component cost.

6. Safeguard policies

Within Section 4, “Safeguard Policies that Might Apply”, extract:

- each explicitly listed safeguard-policy item;
- the currently assessed environmental-assessment category;
- the potential environmental-assessment category described for
  community sub-project grants with environmental implications.

Preserve source codes, labels and represented spellings exactly as
shown. Do not silently repair typographical errors or category codes.

7. Tentative financing

Within Section 5, “Tentative financing”, extract:

- one record for every explicitly represented financing-source row;
- the explicitly represented Total row.

Preserve the source labels exactly as represented.

Use the represented monetary scale as Unit.

Do not calculate or recompute the Total.

8. Contact information

Within Section 6, “Contact point”, extract one record for each labelled
contact field:

- Contact
- Title
- Tel
- Fax
- Email

Preserve phone numbers and email addresses as JSON strings.

For every included record extract exactly these fields:

- Category
- Topic
- Description
- Value
- Unit
- Qualifier
- Reporting Period
- Source Location

Category:

Use exactly one of:

- Project metadata
- Development issue
- Bank rationale
- Project objective
- Project component
- Safeguard policy
- Financing
- Contact information

Topic:

- Preserve the relevant source-grounded project field, issue,
  objective, component, policy, financing source or contact item.
- Do not merge distinct source observations.

Description:

- Provide a concise source-grounded description of the represented
  record.
- Do not add external interpretation.

Value:

- Use a JSON number for explicitly represented numeric values.
- Use a JSON string for explicitly represented textual values, codes,
  dates, telephone numbers, email addresses, textual fractions or
  category labels.
- Use null when no separate Value is represented.
- Preserve textual fractional quantities in their represented textual
  form rather than converting them to numeric percentages.
- Do not derive separate numbers from percentages embedded within a
  complete textual field.
- Do not calculate, infer, derive, rescale or convert values.

Unit:

- Preserve the explicitly associated measurement unit or scale.
- Use null when no explicit unit applies.
- Do not place approximation, threshold or inequality wording in Unit.

Qualifier:

- Preserve explicit source qualifiers or modifiers associated with a
  Value in this field.
- This includes approximation, threshold, extent or ceiling wording
  represented by the source.
- Use null when no explicit qualifier applies.
- Do not merge Qualifier wording into Unit.

Reporting Period:

- Preserve explicitly associated dates or periods.
- Use null when no separate reporting period is explicitly associated
  with the record.

Source Location:

Use concise physical-DOC locations grounded in the source-page
boundaries exposed by the converted representation, for example:

- DOC page 1 — Header
- DOC page 1 — Section 1, Key Development Issues
- DOC page 2 — Section 1, Key Development Issues
- DOC page 2 — Section 1, Rationale for Bank Involvement
- DOC page 3 — Section 2, Proposed objective(s)
- DOC page 3 — Section 3, Preliminary description
- DOC page 4 — Section 4, Safeguard Policies that Might Apply
- DOC page 4 — Section 5, Tentative financing
- DOC page 4 — Section 6, Contact point

Additional extraction rules:

- Use only information explicitly represented in the attached source representation.
- Preserve source wording, codes, labels and spellings where relevant.
- Preserve represented typographical errors rather than silently correcting them.
- Preserve repeated observations if the fixed scope explicitly requires them in distinct source locations.
- Do not use external knowledge.
- Do not follow external links.
- Do not calculate or infer missing information.
- Do not repair source values or codes.
- Do not convert units.
- Do not add explanatory examples from narrative text outside the defined extraction scope.
- Ignore Markdown syntax and page-boundary labels except as structural cues.
- Verify that all content within the defined source scope has been processed.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names and field order defined below.

Expected JSON structure:

{
  "document_id": "D8",
  "branch": "C",
  "records": [
    {
      "Category": null,
      "Topic": null,
      "Description": null,
      "Value": null,
      "Unit": null,
      "Qualifier": null,
      "Reporting Period": null,
      "Source Location": null
    }
  ]
}

Return only the JSON object.
""".strip()


PROMPT_PATH.write_text(
    BRANCH_C_PROMPT,
    encoding="utf-8"
)

PROMPT_SHA256 = sha256_file(
    PROMPT_PATH
)

print(
    "Prompt saved:",
    PROMPT_PATH.name
)

print(
    "Prompt SHA-256:",
    PROMPT_SHA256
)


Prompt saved: D8_branch_C_prompt.txt
Prompt SHA-256: 0f876c0b3992e331801cc6e86828c9f3a52f778e0b0a649aa5ae0e0553137c77


In [16]:
# ============================================================
# 10. Create representation and pre-extraction metadata
# ============================================================

REPRESENTATION_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_format":
        SOURCE_FORMAT,

    "legacy_binary_word_format":
        True,

    "parent_B_representation_file":
        BRANCH_B_REPRESENTATION_PATH.name,

    "parent_B_representation_sha256":
        SOURCE_B_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "parent_B_equivalence_method":
        "Frozen Branch B artefact SHA-256 verification",

    "representation_type":
        "Complete Branch B page-aware structural Markdown with deterministic normalisation",

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "complete_4_page_representation_retained":
        True,

    "scope_enforced_by_prompt_not_representation_filtering":
        True,

    "structural_conversion_inherited_from_branch_B":
        True,

    "structural_financing_label_reconstruction_inherited_from_branch_B":
        True,

    "financing_label_reconstruction_count":
        FINANCING_LABEL_RECONSTRUCTION_COUNT,

    "normalisation_applied":
        True,

    "normalisation_operations": [
        "Unicode NFKC normalisation",
        "Unicode-space standardisation",
        "apostrophe standardisation",
        "dash/minus-glyph standardisation",
        "soft-hyphen removal",
        "line-ending standardisation",
        "horizontal whitespace normalisation",
        "excessive blank-line standardisation"
    ],

    "paragraph_line_merging_applied":
        False,

    "line_break_hyphenation_repair_applied":
        False,

    "semantic_harmonisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "source_spelling_repair_applied":
        False,

    "unit_conversion_applied":
        False,

    "numeric_calculation_applied":
        False,

    "manual_correction_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ]
}


REPRESENTATION_METADATA_PATH.write_text(
    json.dumps(
        REPRESENTATION_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


EXPERIMENT_METADATA_PRE = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_HASH_MATCH,

    "source_format":
        SOURCE_FORMAT,

    "legacy_binary_word_format":
        True,

    "input_representation":
        "Complete deterministically normalised page-aware structural Markdown",

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "parent_B_equivalence_method":
        "Frozen Branch B artefact SHA-256 verification",

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "direct_document_ingestion":
        False,

    "structural_conversion_applied":
        True,

    "structural_conversion_inherited_from_branch_B":
        True,

    "normalisation_applied":
        True,

    "complete_source_document_retained":
        True,

    "source_scope_filtering_applied":
        False,

    "reference_values_disclosed_to_model":
        False,

    "reference_values_used_for_transformation":
        False,

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "manual_response_repair_permitted":
        False,

    "expected_output_format":
        "JSON object",

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "execution_environment":
        "Independent ChatGPT conversation",

    "model":
        "GPT-5.5",

    "created_at":
        datetime.now().isoformat(),

    "python_version":
        sys.version,

    "platform":
        platform.platform(),

    "validation_status":
        "Pending independent Branch C extraction and Stage 4 Validation C"
}


EXPERIMENT_METADATA_PRE_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D8",
  "document_name": "World Bank — Bhutan - Land Management Project — Project Information Document (PID), Concept Stage",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D8 - Project0Inform1ment010Concept0Stage.doc",
  "source_sha256": "61aacfd3138ecfba59fac51d29a970de45d8a909c74b744e756e8a283666c7b5",
  "source_verified": true,
  "source_format": ".doc",
  "legacy_binary_word_format": true,
  "input_representation": "Complete deterministically normalised page-aware structural Markdown",
  "representation_file": "D8_branch_C_normalised_markdown.md",
  "representation_sha256": "9c0d72c972748af919626e4b4d0c47893c2531ecef6e90ac8572efe102f09193",
  "parent_B_equivalence_passed": true,
  "parent_B_equivalence_method": "Frozen Branch B artefact SHA-256 verification",
  "normalisation_integrity_passed": true,
  "direct_document_ingestion": false,
  "structural_conversion_applied": true,
  "structural_conversion_

In [17]:
# ============================================================
# 11. Final pre-extraction control check
# ============================================================
PRECHECK = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "source_identity_verified":
        SOURCE_HASH_MATCH,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "complete_4_page_representation_retained":
        True,

    "page_sequence_preserved":
        page_sequence_preserved,

    "all_section_markers_preserved":
        all_section_markers_preserved,

    "all_source_markers_preserved":
        all_source_markers_preserved,

    "all_financing_labels_preserved":
        all_financing_labels_preserved,

    "representation_exists":
        REPRESENTATION_PATH.exists(),

    "prompt_exists":
        PROMPT_PATH.exists(),

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "reference_values_used_for_transformation":
        False,

    "ready_for_independent_llm_execution": bool(
        SOURCE_HASH_MATCH
        and PARENT_EQUIVALENCE_PASSED
        and normalisation_check[
            "normalisation_integrity_passed"
        ]
        and REPRESENTATION_PATH.exists()
        and PROMPT_PATH.exists()
    )
}


PRECHECK_PATH.write_text(
    json.dumps(
        PRECHECK,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        PRECHECK,
        ensure_ascii=False,
        indent=2
    )
)

if not PRECHECK[
    "ready_for_independent_llm_execution"
]:
    raise ValueError(
        "D8 Branch C is not ready for independent LLM execution."
    )


{
  "document_id": "D8",
  "branch": "C",
  "source_identity_verified": true,
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "complete_4_page_representation_retained": true,
  "page_sequence_preserved": true,
  "all_section_markers_preserved": true,
  "all_source_markers_preserved": true,
  "all_financing_labels_preserved": true,
  "representation_exists": true,
  "prompt_exists": true,
  "expected_record_count_disclosed_to_model": false,
  "expected_category_counts_disclosed_to_model": false,
  "reference_values_used_for_transformation": false,
  "ready_for_independent_llm_execution": true
}


In [18]:
# ============================================================
# 12. Download pre-extraction Branch C artefacts
# ============================================================
for path in [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    REPRESENTATION_METADATA_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    PRECHECK_PATH
]:
    files.download(
        path
    )

print(
    "\nIndependent extraction instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D8_branch_C_normalised_markdown.md.\n"
    "3. Submit D8_branch_C_prompt.txt exactly once.\n"
    "4. Do not upload the original DOC, Branch B artefacts, "
    "Stage 1 reference values, or previous extraction outputs.\n"
    "5. Do not manually repair, correct, or regenerate the response.\n"
    "6. Save the complete response exactly as returned in a plain-text file."
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Independent extraction instructions:
1. Open a new independent ChatGPT conversation.
2. Upload ONLY D8_branch_C_normalised_markdown.md.
3. Submit D8_branch_C_prompt.txt exactly once.
4. Do not upload the original DOC, Branch B artefacts, Stage 1 reference values, or previous extraction outputs.
5. Do not manually repair, correct, or regenerate the response.
6. Save the complete response exactly as returned in a plain-text file.


In [19]:
# ============================================================
# 13. Upload and preserve the complete raw Branch C response
# ============================================================
uploaded_response = files.upload()

if len(uploaded_response) != 1:
    raise ValueError(
        "Upload exactly one complete raw Branch C response file."
    )

RAW_RESPONSE_SOURCE = Path(
    next(
        iter(
            uploaded_response
        )
    )
)

RAW_RESPONSE_TEXT = (
    RAW_RESPONSE_SOURCE.read_text(
        encoding="utf-8"
    )
)

RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)

RAW_RESPONSE_SHA256 = sha256_file(
    RAW_RESPONSE_PATH
)

print(
    "Raw response preserved unchanged."
)

print(
    "Raw response SHA-256:",
    RAW_RESPONSE_SHA256
)


Saving D8_branch_C_raw_response.txt to D8_branch_C_raw_response.txt
Raw response preserved unchanged.
Raw response SHA-256: 5497e30675e43b5dc01ba6056d252a728d0e2128872260005eb97a9237a5fd7b


In [20]:
# ============================================================
# 14. Parse raw response WITHOUT repair
# ============================================================
valid_json = True
json_parsing_error = None
parsed_response = None

try:
    parsed_response = json.loads(
        RAW_RESPONSE_TEXT
    )

except json.JSONDecodeError as exc:
    valid_json = False
    json_parsing_error = str(exc)


top_level_object_valid = (
    valid_json
    and isinstance(
        parsed_response,
        dict
    )
)

document_id_present = (
    top_level_object_valid
    and "document_id"
    in parsed_response
)

document_id_correct = (
    document_id_present
    and parsed_response.get(
        "document_id"
    )
    == DOCUMENT_ID
)

branch_present = (
    top_level_object_valid
    and "branch"
    in parsed_response
)

branch_correct = (
    branch_present
    and parsed_response.get(
        "branch"
    )
    == BRANCH
)

records_present = (
    top_level_object_valid
    and "records"
    in parsed_response
)

records_is_list = (
    records_present
    and isinstance(
        parsed_response.get(
            "records"
        ),
        list
    )
)

records_evaluable = bool(
    valid_json
    and top_level_object_valid
    and document_id_correct
    and branch_correct
    and records_is_list
)

extracted_records = (
    parsed_response["records"]
    if records_evaluable
    else []
)

observed_record_count = (
    len(
        extracted_records
    )
    if records_evaluable
    else None
)

print(
    "Valid JSON:",
    valid_json
)

print(
    "Records evaluable:",
    records_evaluable
)

print(
    "Observed records:",
    observed_record_count
)

if json_parsing_error:
    print(
        "JSON parsing error:",
        json_parsing_error
    )


Valid JSON: True
Records evaluable: True
Observed records: 49


In [21]:
# ============================================================
# 15. Validate record schema and field types
# ============================================================
record_structure_issues = []
field_type_issues = []
missing_mandatory_values = []

if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):
            record_structure_issues.append({
                "record_index":
                    record_index,

                "issue":
                    "Record is not a JSON object"
            })

            continue


        observed_fields = list(
            record.keys()
        )

        if observed_fields != EXPECTED_FIELDS:
            record_structure_issues.append({
                "record_index":
                    record_index,

                "issue":
                    "Field names or field order differ",

                "expected_fields":
                    EXPECTED_FIELDS,

                "observed_fields":
                    observed_fields
            })


        for field in STRING_OR_NULL_FIELDS:

            value = record.get(
                field
            )

            if (
                value is not None
                and not isinstance(
                    value,
                    str
                )
            ):
                field_type_issues.append({
                    "record_index":
                        record_index,

                    "field":
                        field,

                    "observed_type":
                        type(
                            value
                        ).__name__
                })


        value = record.get(
            "Value"
        )

        if (
            isinstance(
                value,
                bool
            )
            or (
                value is not None
                and not isinstance(
                    value,
                    (
                        str,
                        int,
                        float
                    )
                )
            )
        ):
            field_type_issues.append({
                "record_index":
                    record_index,

                "field":
                    "Value",

                "observed_type":
                    type(
                        value
                    ).__name__
            })


        for field in MANDATORY_CONTENT_FIELDS:

            value = record.get(
                field
            )

            if (
                value is None
                or (
                    isinstance(
                        value,
                        str
                    )
                    and not value.strip()
                )
            ):
                missing_mandatory_values.append({
                    "record_index":
                        record_index,

                    "field":
                        field
                })


record_schema_valid = (
    len(
        record_structure_issues
    )
    == 0
    if records_evaluable
    else None
)

field_types_valid = (
    len(
        field_type_issues
    )
    == 0
    if records_evaluable
    else None
)

mandatory_fields_complete = (
    len(
        missing_mandatory_values
    )
    == 0
    if records_evaluable
    else None
)

records_with_type_issues = (
    len({
        issue["record_index"]
        for issue
        in field_type_issues
    })
    if records_evaluable
    else None
)

print(
    "Record schema valid:",
    record_schema_valid
)

print(
    "Field types valid:",
    field_types_valid
)

print(
    "Mandatory fields complete:",
    mandatory_fields_complete
)

print(
    "Structure issues:",
    len(
        record_structure_issues
    )
)

print(
    "Type issues:",
    len(
        field_type_issues
    )
)


Record schema valid: True
Field types valid: True
Mandatory fields complete: True
Structure issues: 0
Type issues: 0


In [22]:
# ============================================================
# 16. Content/scope diagnostics kept separate from schema validity
# ============================================================
if records_evaluable:

    record_count_valid = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )

    observed_category_counts = dict(
        Counter(
            record.get(
                "Category"
            )
            for record
            in extracted_records
            if isinstance(
                record,
                dict
            )
        )
    )

    categories_valid = set(
        observed_category_counts
    ).issubset(
        ALLOWED_CATEGORIES
    )

    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )


    # Exact complete-record duplicate diagnostic only.
    duplicate_counter = Counter(
        tuple(
            json.dumps(
                record.get(field),
                ensure_ascii=False,
                sort_keys=True
            )
            for field
            in EXPECTED_FIELDS
        )
        for record
        in extracted_records
        if isinstance(
            record,
            dict
        )
    )

    duplicate_complete_records = [
        {
            "record":
                list(
                    key
                ),

            "occurrence_count":
                count
        }

        for key, count
        in duplicate_counter.items()

        if count > 1
    ]

    duplicate_complete_record_count = len(
        duplicate_complete_records
    )


    observed_qualifier_values = sorted({
        record.get(
            "Qualifier"
        )

        for record
        in extracted_records

        if (
            isinstance(
                record,
                dict
            )
            and isinstance(
                record.get(
                    "Qualifier"
                ),
                str
            )
        )
    })


    qualifier_presence = {
        qualifier:
            any(
                qualifier.casefold()
                in observed_qualifier.casefold()

                for observed_qualifier
                in observed_qualifier_values
            )

        for qualifier
        in EXPECTED_QUALIFIER_VALUES
    }


    expected_qualifiers_preserved = all(
        qualifier_presence.values()
    )


    qualifier_embedded_in_unit_records = [
        {
            "record_index":
                record_index,

            "Unit":
                record.get(
                    "Unit"
                )
        }

        for record_index, record
        in enumerate(
            extracted_records
        )

        if (
            isinstance(
                record,
                dict
            )
            and isinstance(
                record.get(
                    "Unit"
                ),
                str
            )
            and any(
                qualifier.casefold()
                in record.get(
                    "Unit"
                ).casefold()

                for qualifier
                in EXPECTED_QUALIFIER_VALUES
            )
        )
    ]


    extracted_text_values_casefold = {
        record.get(
            "Value"
        ).strip().casefold()

        for record
        in extracted_records

        if (
            isinstance(
                record,
                dict
            )
            and isinstance(
                record.get(
                    "Value"
                ),
                str
            )
        )
    }


    textual_quantity_checks = {
        "one-quarter":
            "one-quarter"
            in extracted_text_values_casefold,

        "one-third":
            "one-third"
            in extracted_text_values_casefold
    }


    textual_quantities_preserved = all(
        textual_quantity_checks.values()
    )


    extraction_text = json.dumps(
        extracted_records,
        ensure_ascii=False
    )


    source_typo_checks = {
        marker:
            marker
            in extraction_text

        for marker
        in [
            "Indigenous Pelple",
            "F1",
            "BORROWER/RECEPIENT"
        ]
    }


    source_typos_preserved = all(
        source_typo_checks.values()
    )


    financing_total_present = any(
        (
            isinstance(
                record,
                dict
            )
            and record.get(
                "Category"
            )
            == "Financing"
            and str(
                record.get(
                    "Topic"
                )
            ).casefold()
            == "total"
            and record.get(
                "Value"
            )
            == 16.5
        )
        for record
        in extracted_records
    )


else:

    record_count_valid = None
    observed_category_counts = None
    categories_valid = None
    category_counts_valid = None
    duplicate_complete_records = None
    duplicate_complete_record_count = None
    observed_qualifier_values = None
    qualifier_presence = None
    expected_qualifiers_preserved = None
    qualifier_embedded_in_unit_records = None
    textual_quantity_checks = None
    textual_quantities_preserved = None
    source_typo_checks = None
    source_typos_preserved = None
    financing_total_present = None


scope_complete = bool(
    record_count_valid
    and category_counts_valid
) if records_evaluable else False


CONTENT_DIAGNOSTICS = {
    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches_reference":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "categories_valid":
        categories_valid,

    "category_counts_match_reference":
        category_counts_valid,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "missing_mandatory_value_count":
        (
            len(
                missing_mandatory_values
            )
            if records_evaluable
            else None
        ),

    "duplicate_complete_record_count":
        duplicate_complete_record_count,

    "duplicate_complete_records":
        duplicate_complete_records,

    "duplicate_check_is_diagnostic_only":
        True,

    "observed_qualifier_values":
        observed_qualifier_values,

    "expected_qualifier_presence":
        qualifier_presence,

    "expected_qualifiers_preserved":
        expected_qualifiers_preserved,

    "qualifier_embedded_in_unit_count":
        (
            len(
                qualifier_embedded_in_unit_records
            )
            if records_evaluable
            else None
        ),

    "textual_quantity_checks":
        textual_quantity_checks,

    "textual_quantities_preserved":
        textual_quantities_preserved,

    "source_typo_checks":
        source_typo_checks,

    "source_typos_preserved":
        source_typos_preserved,

    "financing_total_present":
        financing_total_present
}


print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)


{
  "expected_record_count": 49,
  "observed_record_count": 49,
  "record_count_matches_reference": true,
  "expected_category_counts": {
    "Project metadata": 13,
    "Development issue": 10,
    "Bank rationale": 2,
    "Project objective": 3,
    "Project component": 3,
    "Safeguard policy": 6,
    "Financing": 7,
    "Contact information": 5
  },
  "observed_category_counts": {
    "Project metadata": 13,
    "Development issue": 10,
    "Bank rationale": 2,
    "Project objective": 3,
    "Project component": 3,
    "Safeguard policy": 6,
    "Financing": 7,
    "Contact information": 5
  },
  "categories_valid": true,
  "category_counts_match_reference": true,
  "mandatory_fields_complete": true,
  "missing_mandatory_value_count": 0,
  "duplicate_complete_record_count": 0,
  "duplicate_complete_records": [],
  "duplicate_check_is_diagnostic_only": true,
  "observed_qualifier_values": [
    "another",
    "at least",
    "could be",
    "less than",
    "now",
    "now assesse

In [23]:
# ============================================================
# 17. Determine technical/schema validity
# ============================================================
#
# Count agreement and content diagnostics are NOT conditions
# for technical/schema validity.
# ============================================================

structure_valid = bool(
    valid_json
    and top_level_object_valid
    and document_id_correct
    and branch_correct
    and records_is_list
    and record_schema_valid is True
    and field_types_valid is True
)


STRUCTURE_CHECK = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        "Complete deterministically normalised page-aware structural Markdown",

    "valid_json":
        bool(
            valid_json
        ),

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        bool(
            top_level_object_valid
        ),

    "document_id_present":
        bool(
            document_id_present
        ),

    "document_id_correct":
        bool(
            document_id_correct
        ),

    "branch_present":
        bool(
            branch_present
        ),

    "branch_correct":
        bool(
            branch_correct
        ),

    "records_present":
        bool(
            records_present
        ),

    "records_is_list":
        bool(
            records_is_list
        ),

    "records_evaluable":
        bool(
            records_evaluable
        ),

    "record_schema_valid":
        record_schema_valid,

    "record_structure_issues":
        (
            record_structure_issues
            if records_evaluable
            else None
        ),

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "field_type_issues":
        (
            field_type_issues
            if records_evaluable
            else None
        ),

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "structure_valid":
        bool(
            structure_valid
        ),

    "scope_complete":
        bool(
            scope_complete
        )
}


STRUCTURE_CHECK_PATH.write_text(
    json.dumps(
        STRUCTURE_CHECK,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        STRUCTURE_CHECK,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D8",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "input_representation": "Complete deterministically normalised page-aware structural Markdown",
  "valid_json": true,
  "json_parsing_error": null,
  "top_level_object_valid": true,
  "document_id_present": true,
  "document_id_correct": true,
  "branch_present": true,
  "branch_correct": true,
  "records_present": true,
  "records_is_list": true,
  "records_evaluable": true,
  "record_schema_valid": true,
  "record_structure_issues": [],
  "field_types_valid": true,
  "records_with_type_issues": 0,
  "field_type_issues": [],
  "content_diagnostics": {
    "expected_record_count": 49,
    "observed_record_count": 49,
    "record_count_matches_reference": true,
    "expected_category_counts": {
      "Project metadata": 13,
      "Development issue": 10,
      "Bank rationale": 2,
      "Project objective": 3,
      "Project component": 3,
      "Safeguard policy": 6,
      "Financing": 7,
      

In [24]:
# ============================================================
# 18. Preserve parsed extraction only when records are evaluable
# ============================================================
parsed_extraction_created = False
parsed_extraction_sha256 = None

if records_evaluable:

    canonical_extraction = {
        "document_id":
            DOCUMENT_ID,

        "branch":
            BRANCH,

        "records":
            extracted_records
    }


    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            canonical_extraction,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )


    parsed_extraction_sha256 = sha256_file(
        PARSED_EXTRACTION_PATH
    )

    parsed_extraction_created = True


    print(
        "Parsed extraction saved:",
        PARSED_EXTRACTION_PATH.name
    )

else:

    print(
        "No parsed extraction created because the preserved raw "
        "response does not contain an evaluable records structure."
    )


Parsed extraction saved: D8_branch_C_parsed_extraction.json


In [25]:
# ============================================================
# 19. Create final experiment metadata and summary
# ============================================================
EXPERIMENT_METADATA = {
    **EXPERIMENT_METADATA_PRE,

    "raw_response_file":
        RAW_RESPONSE_PATH.name,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if parsed_extraction_created
            else None
        ),

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "observed_record_count":
        observed_record_count,

    "observed_category_counts":
        observed_category_counts,

    "structure_check_file":
        STRUCTURE_CHECK_PATH.name,

    "structure_valid":
        bool(
            structure_valid
        ),

    "notes": (
        "Branch C applies deterministic normalisation to the exact "
        "complete Branch B page-aware structural Markdown representation. "
        "The Branch B DOC-to-PDF-to-Markdown conversion and its two "
        "source-grounded financing-label continuation reconstructions are "
        "inherited unchanged. No content is filtered by extraction scope. "
        "Stage 1 reference values and expected counts are not supplied to "
        "the model. Accuracy is evaluated separately in Validation C."
    )
}


EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_HASH_MATCH,

    "input_representation":
        "Complete deterministically normalised page-aware structural Markdown",

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "structural_conversion_inherited_from_branch_B":
        True,

    "normalisation_applied":
        True,

    "complete_source_document_retained":
        True,

    "source_scope_filtering_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "raw_response_preserved":
        True,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "valid_json":
        bool(
            valid_json
        ),

    "records_evaluable":
        bool(
            records_evaluable
        ),

    "record_schema_valid":
        record_schema_valid,

    "field_types_valid":
        field_types_valid,

    "structure_valid":
        bool(
            structure_valid
        ),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "scope_complete":
        bool(
            scope_complete
        ),

    "duplicate_complete_record_count":
        duplicate_complete_record_count,

    "expected_qualifiers_preserved":
        expected_qualifiers_preserved,

    "textual_quantities_preserved":
        textual_quantities_preserved,

    "source_typos_preserved":
        source_typos_preserved,

    "financing_total_present":
        financing_total_present,

    "parsed_extraction_created":
        parsed_extraction_created,

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "accuracy_validation_completed":
        False,

    "validation_status":
        (
            "Pending Stage 4 Branch C validation against the fixed Stage 1 "
            "reference dataset using Branch A-frozen D8 comparison rules"
            if records_evaluable
            else
            "Not content-evaluable because the preserved Branch C response "
            "does not contain an evaluable records structure"
        )
}


EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D8",
  "document_name": "World Bank — Bhutan - Land Management Project — Project Information Document (PID), Concept Stage",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D8 - Project0Inform1ment010Concept0Stage.doc",
  "source_sha256": "61aacfd3138ecfba59fac51d29a970de45d8a909c74b744e756e8a283666c7b5",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised page-aware structural Markdown",
  "representation_file": "D8_branch_C_normalised_markdown.md",
  "representation_sha256": "9c0d72c972748af919626e4b4d0c47893c2531ecef6e90ac8572efe102f09193",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "structural_conversion_inherited_from_branch_B": true,
  "normalisation_applied": true,
  "complete_source_document_retained": true,
  "source_scope_filtering_applied": false,
  "reference_values_used_for_transformation": false,
  "expected_record_

In [26]:
# ============================================================
# 20. Final artefact inventory and download
# ============================================================
artefacts = [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    REPRESENTATION_METADATA_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    PRECHECK_PATH,
    RAW_RESPONSE_PATH,
    STRUCTURE_CHECK_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]

if parsed_extraction_created:
    artefacts.append(
        PARSED_EXTRACTION_PATH
    )

print(
    "Final D8 Branch C artefacts:"
)

for path in artefacts:

    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )


for path in artefacts:

    if path.exists():
        files.download(
            path
        )


Final D8 Branch C artefacts:
- D8_branch_C_parent_B_equivalence_check.json | exists: True
- D8_branch_C_normalisation_check.json | exists: True
- D8_branch_C_normalised_markdown.md | exists: True
- D8_branch_C_representation_metadata.json | exists: True
- D8_branch_C_prompt.txt | exists: True
- D8_branch_C_experiment_metadata_pre.json | exists: True
- D8_branch_C_pre_extraction_check.json | exists: True
- D8_branch_C_raw_response.txt | exists: True
- D8_branch_C_structure_check.json | exists: True
- D8_branch_C_experiment_metadata.json | exists: True
- D8_branch_C_experiment_summary.json | exists: True
- D8_branch_C_parsed_extraction.json | exists: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>